# Adversarial robustness of a toxicity classifier

This notebook evaluates how `unitary/toxic-bert` behaves when comment text is obfuscated in ways that a person can still read.

**Context.** Automated moderation runs at a threshold chosen on clean data, and people posting abusive content do not send clean data. The useful question is how much of the model's measured performance survives someone trying to get around it, and what it costs to close whatever gap exists.

**Scope.** Six transformations, all of them a few lines of string manipulation. No model access, no gradients, no machine learning on the attacker's side. If the model fails against these it will fail against anything more sophisticated.

**Both directions are measured.** Toxic content that gets through and benign content that is wrongly flagged are separate failures with different costs. Scoring only toxic examples hides the second one, which turned out to be the larger problem here.

The same run is available headless via `python scripts/run_experiment.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from evasion_gap.attacks import ATTACKS
from evasion_gap.data import load_corpus
from evasion_gap.defense import normalize_text
from evasion_gap.model import Scorer
from evasion_gap.pipeline import build_operating_point, run_sweep
from evasion_gap.plots import plot_defense_effect, plot_recall_by_threshold

config = yaml.safe_load((ROOT / "config.yaml").read_text())
config

## 1. Data

`civil_comments`, streamed and filtered to human-rated toxicity of at least 0.8 for the toxic split and at most 0.1 for the benign split.

The first run takes about five minutes because comments above the toxicity cutoff are rare and the stream has to read a lot of rows. The result is cached to `data/`, so later runs are immediate.

In [ ]:
toxic, benign = load_corpus(**config["dataset"])
print(f"{len(toxic)} toxic, {len(benign)} benign")
print(f"\nexample toxic comment:\n{toxic[0][:200]}")

In [ ]:
lengths = pd.DataFrame({
    "split": ["toxic"] * len(toxic) + ["benign"] * len(benign),
    "chars": [len(t) for t in toxic] + [len(b) for b in benign],
    "words": [len(t.split()) for t in toxic] + [len(b.split()) for b in benign],
})
lengths.groupby("split")[["chars", "words"]].describe().T

## 2. Clean baseline

Score both splits before touching anything. The shape of these distributions determines the thresholds, and one feature of them explains a mistake made in the first version of this analysis.

In [ ]:
scorer = Scorer(
    model_id=config["model_id"],
    batch_size=config["eval"]["batch_size"],
    max_length=config["eval"]["max_length"],
)

toxic_scores = scorer(toxic)
benign_scores = scorer(benign)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(benign_scores, bins=40, alpha=0.6, label="benign")
ax.hist(toxic_scores, bins=40, alpha=0.6, label="toxic")
ax.set_xlabel("P(toxic)")
ax.set_ylabel("count")
ax.set_title("Score distribution on clean text")
ax.legend()
plt.tight_layout()

A substantial number of genuinely toxic comments score near zero. That lower tail is why a threshold targeting high recall ends up very close to zero, which is covered in the next cell.

## 3. Thresholds

Two thresholds, because they give different answers:

- `high_recall`: set to catch 95 percent of clean toxic comments. Common in write-ups.
- `fpr_1pct`: set to keep false positives on benign comments at 1 percent. Closer to how a moderation system is configured, because wrongly removing content from users who have done nothing wrong is the expensive error.

In [ ]:
operating_points = [
    build_operating_point(spec, toxic_scores, benign_scores)
    for spec in config["eval"]["operating_points"]
]
pd.DataFrame([op.as_dict() for op in operating_points])

The recall-pinned threshold lands at about 0.02 and carries a 7 percent false positive rate. No moderation system would run at that setting, and a rate measured there barely responds to changes in the input. Results from that threshold are reported below for comparison, but the 1 percent budget is the one the conclusions are drawn from.

## 4. The transformations

Each one has to leave the text readable. Something that destroys the text is not an obfuscation technique, because a real poster needs their message to land.

In [ ]:
sample = toxic[0][:80]
for name, fn in ATTACKS.items():
    print(f"{name:>12}: {fn(sample)}")

## 5. Sweep

Every combination of split, transformation and defense condition, scored at both thresholds. Thresholds stay fixed throughout, since a threshold chosen on clean data is what a deployed system would be using.

This is the slow cell: 2 splits x 7 transformations x 2 defense conditions x 300 comments.

In [ ]:
sweep = run_sweep(
    scorer,
    {"toxic": toxic, "benign": benign},
    operating_points,
    seed=config["seed"],
)
sweep.head()

In [ ]:
shipping = config["eval"]["reporting_operating_point"]

def table(split):
    sub = sweep[(sweep["split"] == split) & (sweep["operating_point"] == shipping)]
    return sub.pivot(index="attack", columns="defense", values="rate").sort_values("none")

print("recall on toxic content")
display(table("toxic"))
print("\nfalse positive rate on benign content")
display(table("benign"))

## 6. Reading the two tables together

The toxic table on its own suggests that only `homoglyph` is a problem and that the model handles everything else. The benign table shows that reading is wrong.

`spaced` raises the false positive rate on ordinary comments from 1 percent to 99 percent. `repeated` reaches 96 percent, `leetspeak` 74 percent, `devowel` 71 percent. Those transformations were never evading the model. They push its score up regardless of what the comment says, which is why recall looked fine and why the same inputs are catastrophic on benign text.

So there are two distinct failures:

1. `homoglyph` is a genuine evasion. It lowers the score on toxic content.
2. `spaced`, `repeated`, `leetspeak` and `devowel` are false positive triggers. They raise the score on everything.

These need different fixes, which is the subject of the rest of the notebook.

## 7. Why homoglyph works, and why zero_width does not

Both transformations change the bytes without changing what a reader sees, but only one changes the model's output. The tokenizer explains the difference.

In [ ]:
probe = "you are an idiot"
for name in ["clean", "zero_width", "homoglyph"]:
    tokens = scorer.tokenizer.tokenize(ATTACKS[name](probe))
    print(f"{name:>11}: {tokens}")

`zero_width` produces exactly the same tokens as clean text. BERT's tokenizer removes Unicode category Cf characters during text cleaning, so the inserted characters never reach the model. That protection is real but incidental: it comes from a preprocessing detail, not from a decision anyone made about obfuscation, and a different tokenizer would not necessarily keep it.

`homoglyph` breaks the tokens apart. Cyrillic characters are different codepoints, so the affected words fall out of vocabulary and the model scores text it has effectively never seen.

## 8. Does normalization fix it?

The normalization pass strips category Cf characters, maps the known confusable codepoints back to Latin, and applies NFKC. It runs before the model, so it needs no retraining and can be rolled back on its own.

In [ ]:
for name in ["homoglyph", "zero_width", "spaced"]:
    before = ATTACKS[name](probe)
    print(f"{name:>11}: {before!r}\n{'':>13}-> {normalize_text(before)!r}\n")

In [ ]:
fig, axes = plot_defense_effect(sweep, shipping, outfile=ROOT / "results" / "defense_effect.png")

Normalization restores homoglyph recall from 0.303 to 0.780, matching the clean baseline exactly, and brings its benign false positive rate back from 0.207 to 0.010.

It does nothing for `spaced`, `repeated`, `leetspeak` or `devowel`, and it should not be expected to. Those inputs are not disguising anything, so there is nothing to undo. Fixing them means changing what the model learned, not what it is fed.

## 9. Threshold choice

The same measurement at both thresholds, to show how much the conclusion depends on where the threshold sits.

In [ ]:
fig, axes = plot_recall_by_threshold(
    sweep, operating_points, outfile=ROOT / "results" / "recall_by_threshold.png"
)

In [ ]:
comparison = (
    sweep[(sweep["split"] == "toxic") & (sweep["defense"] == "none")]
    .pivot(index="attack", columns="operating_point", values="delta")
    .rename(columns=lambda c: f"recall drop at {c}")
)
comparison.loc[["homoglyph", "leetspeak", "devowel", "zero_width"]]

The homoglyph attack reads as a 0.017 recall drop at the recall-pinned threshold and a 0.477 drop at the 1 percent budget. Same attack, same data, same model.

The first version of this analysis used only the recall-pinned threshold and concluded the model was robust. That was wrong. The signal that something was off was that homoglyph showed a 0.017 recall drop while its mean score fell from 0.605 to 0.214, which is a large change in the model's output that the metric was not registering.

## 10. Conclusions

**Homoglyph substitution defeats the model and is fully fixable in preprocessing.** Recall falls from 0.780 to 0.303, a 61 percent relative loss, and normalization restores it completely at negligible cost.

**The model reacts to character patterns rather than content.** Four transformations raise the false positive rate on ordinary comments to between 71 and 99 percent. This is a larger practical problem than the evasion, it affects users who have done nothing wrong, and preprocessing does not address it. It needs benign training examples that use emphatic formatting.

**Evaluation design determined what was visible.** Neither finding shows up under a recall-pinned threshold measured on toxic examples alone, which is the setup used in the first version of this work.

### Recommended next actions

1. Ship the normalization pass. It is cheap, it is testable, and it closes the evasion.
2. Retrain with emphatically formatted benign examples, then re-run this sweep to check whether the false positive behaviour improves.
3. Set evaluation thresholds from the false positive budget and always score both splits.

### Limitations

One model, one dataset, English only. 300 examples per condition gives roughly plus or minus 4 percentage points on each rate. The transformations are hand-written rather than searched, so the recall loss is a lower bound. The `civil_comments` labels are crowd-sourced and their annotator bias propagates into the thresholds, which are derived from that labelling.